# Import from Excel

In [ ]:
import pandas as pd
df = pd.read_excel('~/Documents/Arkitektur Alle i Tjeneste 2025.xlsx')
df.head()

## 1. Read Groups

In [ ]:
# Add root node
groups = pd.DataFrame([{'ID': 1, 'name': 'BCC København', 'parentID': None}])

# Add L1 groups
l1_groups = df[['L1']].drop_duplicates().reset_index(drop=True)
l1_groups.rename(columns={'L1': 'name'}, inplace=True)
l1_groups['ID'] = l1_groups.index + 2
l1_groups['parentID'] = 1
groups = pd.concat([groups, l1_groups], ignore_index=True)

# Add L2 groups
l2_groups = df[['L1', 'L2']].dropna().drop_duplicates().reset_index(drop=True)
l2_groups = l2_groups.merge(groups[['ID', 'name']], left_on='L1', right_on='name', how='left')
l2_groups = l2_groups.drop(columns=['name', 'L1'])
l2_groups = l2_groups.rename(columns={'L2': 'name', 'ID': 'parentID'})
l2_groups['ID'] = l2_groups.index + 1 + groups['ID'].max()
groups = pd.concat([groups, l2_groups], ignore_index=True)

# Add L3 groups
l3_groups = df[['L2', 'L3']].dropna().drop_duplicates().reset_index(drop=True)
l3_groups = l3_groups.merge(groups[['ID', 'name']], left_on='L2', right_on='name', how='left')
l3_groups = l3_groups.drop(columns=['name', 'L2'])
l3_groups = l3_groups.rename(columns={'L3': 'name', 'ID': 'parentID'})
l3_groups['ID'] = l3_groups.index + 1 + groups['ID'].max()
groups = pd.concat([groups, l3_groups], ignore_index=True)

# Add L4 groups
l4_groups = df[['L3', 'L4']].dropna().drop_duplicates().reset_index(drop=True)
l4_groups = l4_groups.merge(groups[['ID', 'name']], left_on='L3', right_on='name', how='left')
l4_groups = l4_groups.drop(columns=['name', 'L3'])
l4_groups = l4_groups.rename(columns={'L4': 'name', 'ID': 'parentID'})
l4_groups['ID'] = l4_groups.index + 1 + groups['ID'].max()
groups = pd.concat([groups, l4_groups], ignore_index=True)

groups


## 2. Read Members

In [ ]:
names = df[['L1', 'L2', 'L3', 'L4', 'Navn', 'Titel']]
names['Group'] = names['L1']
names.loc[names['L2'].notnull(), 'Group'] = names['L2']
names.loc[names['L3'].notnull(), 'Group'] = names['L3']
names.loc[names['L4'].notnull(), 'Group'] = names['L4']
names = names[['Navn', 'Titel', 'Group']]
names

## 3. Import to Database

In [ ]:
from supabase import Client, create_client
import os
from dotenv import load_dotenv
load_dotenv()

supabase: Client = create_client(
    os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"]
)

### 3.1 Import groups

In [ ]:

# Prepare the data for insertion
records = []
for _, row in groups.iterrows():
    records.append({
        'id': int(row['ID']),
        'name': row['name'],
        'parent_id': int(row['parentID']) if pd.notnull(row['parentID']) else None,
        'tenant_id': 51,
    })

# Insert into Supabase
response = supabase.table('groups').insert(records).execute()
print(response)

### 3.2 Import members

In [ ]:
# Lookup Supabase group IDs for each Group in `names` and add as a new column `group_id`.
# Uses existing `supabase` client and `names` dataframe already defined in the notebook.

# Prepare unique group names
unique_groups = names['Group'].dropna().astype(str).str.strip().unique().tolist()

# Cache for lookups to avoid repeated requests
group_id_cache = {}

def _extract_id_from_resp(resp):
    # Support response objects or dicts returned by supabase client
    data = None
    if resp is None:
        return None
    if hasattr(resp, "data"):
        data = resp.data
    elif isinstance(resp, dict):
        data = resp.get("data")
    if data and isinstance(data, list) and len(data) > 0:
        first = data[0]
        if isinstance(first, dict):
            return first.get("id")
    return None

for g in unique_groups:
    if not g:
        group_id_cache[g] = None
        continue
    try:
        resp = supabase.table('groups').select('id').eq('name', g).limit(1).execute()
        gid = _extract_id_from_resp(resp)
        group_id_cache[g] = gid
    except Exception as e:
        # On error, store None and continue
        group_id_cache[g] = None

# Map cached ids back to the dataframe. Use pandas nullable integer dtype.
names['group_id'] = names['Group'].astype(str).str.strip().map(group_id_cache)
try:
    names['group_id'] = names['group_id'].astype('Int64')
except Exception:
    # If conversion fails for any reason, keep as-is (likely object dtype)
    pass

# Summary of results
missing = names['group_id'].isna().sum()
print(f"Resolved group IDs for {len(unique_groups)} unique groups. Rows with missing group_id: {missing}")

# Show a quick preview
names.head()

#### Initialize BCC API Client

In [ ]:
import os
from requests_oauth2client import OAuth2Client, OAuth2ClientCredentialsAuth
import swagger_client as bcc_api_client

# Initialize OAuth2 client for BCC and client-credentials auth (same as backend/app.py)
bcc_oauth_client = OAuth2Client(
    token_endpoint="https://login.bcc.no/oauth/token",
    client_id=os.environ["BCC_OAUTH_CLIENT_ID"],
    client_secret=os.environ["BCC_OAUTH_CLIENT_SECRET"],
)

bcc_auth = OAuth2ClientCredentialsAuth(
    bcc_oauth_client,
    scope="persons.name#read",
    audience="api.bcc.no",
)

# Configure API client
bcc_client_config = bcc_api_client.Configuration()
bcc_client_config.host = "https://core.api.bcc.no"
api_client = bcc_api_client.ApiClient(configuration=bcc_client_config)
persons_api = bcc_api_client.PersonsApi(api_client)


In [ ]:
import json

from swagger_client.models.person import Person

if bcc_auth.token is None or bcc_auth.token.is_expired():
    bcc_auth.renew_token()
persons_api.api_client.configuration.access_token = str(bcc_auth.token)

# Get a list of unique names from the `names` dataframe
unique_names = names['Navn'].dropna().astype(str).str.strip().unique().tolist()

# Look up names in BCC API in groups of 10:
name_uid_map = {}
for i in range(0, len(unique_names), 10):
    batch = unique_names[i:i+10]
    persons: list[Person] = persons_api.find_persons(
        fields="*",
        filter=json.dumps({"displayName": {"_in": batch}}),
    ).data
    for person in persons:
        name_uid_map[person.display_name.strip()] = person.uid

# Add a `uid` column to the `names` dataframe based on the lookups
names['uid'] = names['Navn'].astype(str).str.strip().map(name_uid_map)

In [ ]:
# Print names that were not found (missing uid)
missing_uids = names[~names['Navn'].isna() & names['uid'].isna()]
print(len(missing_uids), "Names not found in BCC API:")
print(missing_uids['Navn'].unique().tolist())

In [ ]:
# Fetch all persons from BCC API (displayName and uid) into a DataFrame.
# Reuses existing `bcc_auth` and `persons_api` from the notebook.

if bcc_auth.token is None or bcc_auth.token.is_expired():
    bcc_auth.renew_token()
persons_api.api_client.configuration.access_token = str(bcc_auth.token)

all_members = []
limit = 500
offset = 0

while True:
    try:
        resp = persons_api.find_persons(fields="*", limit=limit, offset=offset)
    except Exception as e:
        print("Error fetching persons:", e)
        break

    data = getattr(resp, "data", resp)
    if not data:
        break

    for p in data:
        # support object or dict response shapes
        if hasattr(p, "display_name"):
            name = p.display_name
        elif hasattr(p, "displayName"):
            name = p.displayName
        elif isinstance(p, dict):
            name = p.get("display_name") or p.get("displayName")
        else:
            name = None

        if hasattr(p, "uid"):
            uid = p.uid
        elif isinstance(p, dict):
            uid = p.get("uid")
        else:
            uid = None

        all_members.append({"displayName": (name or "").strip() if name else None, "uid": uid})

    # advance pagination
    offset += limit

# Build DataFrame, dedupe by uid
members_df = pd.DataFrame(all_members)
members_df = members_df.drop_duplicates(subset=["uid"]).reset_index(drop=True)

print(f"Fetched {len(members_df)} unique persons.")
members_df.head()

In [ ]:
import unicodedata
import string
from difflib import SequenceMatcher, get_close_matches
from rapidfuzz import process, fuzz
from collections import defaultdict
import pandas as _pd

# Advanced matching to suggest uids for rows in `names` missing `uid`

# prefer rapidfuzz if available for better fuzzy matching
try:
    _HAS_RAPIDFUZZ = True
except Exception:
    _HAS_RAPIDFUZZ = False

def normalize_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    # remove common titles and punctuation, normalize accents
    titles = ["mr", "mrs", "ms", "dr", "prof", "fr", "herr", "frau"]
    s = s.lower()
    # remove punctuation
    s = s.translate(str.maketrans("", "", string.punctuation))
    # normalize diacritics
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # remove duplicate whitespace
    s = " ".join(s.split())
    # remove titles at start
    parts = s.split()
    if parts and parts[0].rstrip(".") in titles:
        parts = parts[1:]
    return " ".join(parts)

# prepare normalized names for members_df
members_df["display_norm"] = members_df["displayName"].astype(str).map(normalize_name)
# create mapping displayName -> uid (keep last if duplicates)
_display_to_uid = dict(zip(members_df["displayName"], members_df["uid"]))
# also mapping normalized -> list of (orig, uid)
norm_map = defaultdict(list)
for orig, uid, norm in zip(members_df["displayName"], members_df["uid"], members_df["display_norm"]):
    norm_map[norm].append((orig, uid))

# Select missing rows from names
mask_missing = names["uid"].isna() & names["Navn"].notna()
missing_df = names.loc[mask_missing, :].copy()
missing_df["nav_norm"] = missing_df["Navn"].astype(str).map(normalize_name)

# Prepare choices for fuzzy matching
choices_original = members_df["displayName"].tolist()
choices_norm = members_df["display_norm"].tolist()

# Function to score using best available method
def best_matches(query_norm, top_n=3):
    results = []
    if not query_norm:
        return results
    # 1) exact normalized match
    if query_norm in norm_map:
        for orig, uid in norm_map[query_norm]:
            results.append((orig, uid, 100.0))
        # return unique originals (exact)
        seen = set()
        uniq = []
        for orig, uid, score in results:
            if orig not in seen:
                seen.add(orig)
                uniq.append((orig, uid, score))
        return uniq[:top_n]
    # 2) rapidfuzz if available (use normalized choices)
    if _HAS_RAPIDFUZZ:
        # map normalized choices to original display names; build a dict of unique norm->orig (first)
        # process.extract expects choices iterable; use display_norm list directly but want original value back
        extracted = process.extract(
            query_norm,
            {i: n for i, n in enumerate(choices_norm)},
            scorer=fuzz.token_set_ratio,
            limit=top_n * 4,
        )
        # extracted entries are (norm_value, score, index)
        seen_uids = set()
        for match_norm, score, idx in extracted:
            orig = choices_original[idx]
            uid = _display_to_uid.get(orig)
            if orig not in seen_uids:
                results.append((orig, uid, float(score)))
                seen_uids.add(orig)
                if len(results) >= top_n:
                    break
        return results
    # 3) fallback to difflib: get close matches from original display names and score with SequenceMatcher
    # try both normalized-to-normalized and original-to-original
    cand_norms = get_close_matches(query_norm, choices_norm, n=top_n * 4, cutoff=0.6)
    seen = set()
    for cand in cand_norms:
        for orig, uid in norm_map.get(cand, []):
            if orig in seen:
                continue
            score = SequenceMatcher(None, query_norm, cand).ratio() * 100
            results.append((orig, uid, float(score)))
            seen.add(orig)
            if len(results) >= top_n:
                break
        if len(results) >= top_n:
            break
    # If still not enough, try get_close_matches on original strings as last resort
    if len(results) < top_n:
        cand_orig = get_close_matches(missing_df.loc[missing_df["nav_norm"] == query_norm, "Navn"].iat[0], choices_original, n=top_n * 2, cutoff=0.6) if not missing_df.empty else []
        for orig in cand_orig:
            if orig in seen:
                continue
            uid = _display_to_uid.get(orig)
            score = SequenceMatcher(None, normalize_name(orig), query_norm).ratio() * 100
            results.append((orig, uid, float(score)))
            seen.add(orig)
            if len(results) >= top_n:
                break
    return results

# Run matching and attach suggestions back to names
suggested = []
HIGH_THRESH = 88.0
MEDIUM_THRESH = 75.0

for idx, row in missing_df.iterrows():
    q = row["nav_norm"]
    candidates = best_matches(q, top_n=3)
    # pick top candidate
    if candidates:
        best_orig, best_uid, best_score = candidates[0]
    else:
        best_orig, best_uid, best_score = (None, None, 0.0)
    confidence = "none"
    if best_score >= HIGH_THRESH:
        confidence = "high"
    elif best_score >= MEDIUM_THRESH:
        confidence = "medium"
    elif best_score > 0:
        confidence = "low"
    suggested.append({
        "index": idx,
        "Navn": row["Navn"],
        "Group": row.get("Group"),
        "suggested_name": best_orig,
        "suggested_uid": best_uid,
        "match_score": best_score,
        "confidence": confidence,
        "top_matches": candidates,
    })

suggested_df = _pd.DataFrame(suggested).set_index("index")

# Merge suggestions into the original names DataFrame (without overwriting existing uid)
if not suggested_df.empty:
    names.loc[suggested_df.index, "suggested_uid"] = suggested_df["suggested_uid"].astype(object)
    names.loc[suggested_df.index, "suggested_name"] = suggested_df["suggested_name"].astype(object)
    names.loc[suggested_df.index, "suggested_score"] = suggested_df["match_score"].astype(float)
    names.loc[suggested_df.index, "suggested_confidence"] = suggested_df["confidence"]

# Summary
total_missing = len(missing_df)
matched_high = (suggested_df["match_score"] >= HIGH_THRESH).sum()
matched_medium = ((suggested_df["match_score"] >= MEDIUM_THRESH) & (suggested_df["match_score"] < HIGH_THRESH)).sum()
matched_low = ((suggested_df["match_score"] > 0) & (suggested_df["match_score"] < MEDIUM_THRESH)).sum()
unmatched = total_missing - (matched_high + matched_medium + matched_low)

print(f"Missing rows: {total_missing}")
print(f"High-confidence matches (≥{HIGH_THRESH}%): {matched_high}")
print(f"Medium-confidence matches (≥{MEDIUM_THRESH}%): {matched_medium}")
print(f"Low-confidence matches (<{MEDIUM_THRESH}% but >0): {matched_low}")
print(f"No candidate found: {unmatched}")

# Show a sample of suggestions (high first)
_preview = suggested_df.sort_values("match_score", ascending=False).head(40)
_preview[["Navn", "suggested_name", "suggested_uid", "match_score", "confidence"]]

### Import persons into Supabase

In [ ]:
# Insert names with both uid and group_id into Supabase group_membership
# Uses existing `names` DataFrame and `supabase` client from the notebook.

# Select rows that have both uid and group_id
to_import = names[names['uid'].notna() & names['group_id'].notna()].copy()

if to_import.empty:
    print("No rows to import (no uid/group_id pairs).")
else:
    # Build records for insertion
    records = []
    for _, r in to_import.iterrows():
        try:
            gid = int(r['group_id'])
        except Exception:
            # skip rows where group_id cannot be converted
            continue
        uid = str(r['uid']).strip()
        if not uid:
            continue
        records.append({
            'tenant_id': 51,
            'group_id': gid,
            'bcc_person_uid': uid,
        })

    if not records:
        print("No valid records to insert after cleaning.")
    else:
        # Insert in batches to avoid very large payloads
        batch_size = 100
        inserted = 0
        seen_keys = set()
        for i in range(0, len(records), batch_size):
            batch = records[i:i+batch_size]
            try:
                # dedupe batch by (group_id, bcc_person_uid) to avoid redundant upserts
                unique_batch = []
                for rec in batch:
                    key = (rec.get('group_id'), rec.get('bcc_person_uid'))
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)
                    unique_batch.append(rec)

                if not unique_batch:
                    print(f"Skipped empty/duplicate batch {i//batch_size + 1}.")
                    continue

                # Upsert using the composite key so existing (group_id, bcc_person_uid) rows are updated instead of duplicated.
                # Replace with on_conflict columns matching your DB unique constraint on (group_id, bcc_person_uid).
                resp = supabase.table('group_membership').upsert(unique_batch, on_conflict="group_id,bcc_person_uid").execute()
                # If resp.data is list of inserted rows, count them; otherwise fallback to batch length
                added = len(resp.data) if hasattr(resp, "data") and isinstance(resp.data, list) else len(batch)
                inserted += added
                print(f"Inserted batch {i//batch_size + 1}: {added} rows.")
            except Exception as e:
                print(f"Error inserting batch {i//batch_size + 1}: {e}")

        print(f"Import complete. Total attempted: {len(records)}. Total inserted (reported): {inserted}.")

In [ ]:
# Update `uid` column from `suggested_uid` where confidence is high
high_conf_mask = names["suggested_confidence"] == "high"
names.loc[high_conf_mask, "uid"] = names.loc[high_conf_mask, "suggested_uid"]